# Phase 12: Executive Dashboard

This notebook documents the executive dashboard data model, KPI definitions, failure trends, station heatmap, bottleneck analytics, SHAP explanations, and business-impact scenario. The dashboard uses the SQLite database created in Phase 11.

## 1. Executive measurement model

| KPI family | Dashboard measure | Interpretation |
|---|---|---|
| Quality | Historical failure rate | Observed target rate in labeled training data |
| Volume | Test products scored and model alerts | Unlabeled production-like population processed by the Phase 6 model |
| Model quality | MCC, precision, recall | Held-out validation performance of the production-safe LightGBM model |
| Process | Waiting time and bottleneck score | Timestamp-derived process constraints |
| Drivers | Mean absolute SHAP | Global model influence, not physical causality |
| Value | Net estimated impact | User-controlled scenario, not realized savings |

In [ ]:
from pathlib import Path
import sqlite3
import sys
import pandas as pd
import plotly.express as px

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

DATABASE_PATH = PROJECT_ROOT / 'data' / 'database' / 'manufacturing_copilot.db'
DATABASE_PATH

## 2. Prepare Phase 12 tables

The preparation script adds ordered failure-time and duration tables plus a compact executive KPI baseline. It does not rebuild or delete the Phase 11 database.

In [ ]:
from src.data.phase12_executive_dashboard import prepare_dashboard_tables, write_report

phase12_summary = prepare_dashboard_tables()
write_report(phase12_summary)
phase12_summary

In [ ]:
connection = sqlite3.connect(DATABASE_PATH)
kpis = pd.read_sql_query('SELECT * FROM executive_kpi_baseline', connection)
display(kpis)

## 3. Failure trend

Bosch timestamps are relative production-time values rather than calendar dates. Therefore the trend is presented as ordered relative production periods (`P01`, `P02`, and so on). This supports seasonality and drift discussion without pretending that the bins are months or weeks.

In [ ]:
trend = pd.read_sql_query(
    'SELECT * FROM failure_time_trends ORDER BY period_order', connection
)
trend['period'] = trend['period_order'].map(lambda value: f'P{value:02d}')
fig = px.line(
    trend,
    x='period',
    y='failure_rate_pct',
    markers=True,
    labels={'period': 'Relative production period', 'failure_rate_pct': 'Failure rate (%)'},
    title='Failure rate across relative production time',
)
fig.show()
display(trend)

## 4. Station heatmap

The heatmap places production line on the vertical axis and station on the horizontal axis. Missing cells indicate that a station is not part of that production line.

In [ ]:
stations = pd.read_sql_query(
    'SELECT line, station, station_key, failure_rate_pct, failure_rate_lift, part_count FROM station_failure_rates',
    connection,
)
heatmap = stations.pivot(index='line', columns='station', values='failure_rate_pct')
heatmap.index = [f'L{value}' for value in heatmap.index]
heatmap.columns = [f'S{int(value)}' for value in heatmap.columns]
fig = px.imshow(
    heatmap,
    color_continuous_scale='YlOrRd',
    labels={'x': 'Station', 'y': 'Production line', 'color': 'Failure rate (%)'},
    title='Station failure-rate heatmap',
)
fig.show()
display(stations.sort_values('failure_rate_pct', ascending=False).head(15))

## 5. Bottleneck analytics

The bottleneck score combines waiting-time, volume, and quality-related evidence from Phase 8. It is a prioritization score rather than a physical unit.

In [ ]:
bottlenecks = pd.read_sql_query(
    'SELECT * FROM bottlenecks ORDER BY bottleneck_rank', connection
)
fig = px.scatter(
    bottlenecks,
    x='avg_waiting_time',
    y='bottleneck_score',
    size='product_count',
    color='failure_rate_pct',
    hover_name='station',
    color_continuous_scale='YlOrRd',
    title='Volume, waiting time, and bottleneck priority',
)
fig.show()
display(bottlenecks.head(15))

## 6. SHAP explanations

Mean absolute SHAP ranks features by their average influence on model predictions. It does not show whether a feature is a confirmed physical cause. Recommended actions should be validated against maintenance, sensor, quality, and process records.

In [ ]:
drivers = pd.read_sql_query(
    'SELECT * FROM failure_drivers ORDER BY driver_rank LIMIT 15', connection
)
fig = px.bar(
    drivers.sort_values('mean_abs_shap'),
    x='mean_abs_shap',
    y='feature',
    orientation='h',
    color='driver_type',
    title='Global failure-prediction drivers',
)
fig.show()
display(drivers)

## 7. Business-impact scenario

The financial calculation is transparent:

1. `expected failures = production volume × historical failure rate`
2. `expected alerts = production volume × deployment alert rate`
3. `true-positive alerts = min(expected failures, expected alerts × validation precision)`
4. `prevented failures = true-positive alerts × intervention effectiveness`
5. `net impact = prevented failures × cost per failure - alerts × review cost`

This is an illustrative scenario. It must be replaced with client-specific costs and measured intervention effectiveness.

In [ ]:
from src.dashboard.business_impact import calculate_business_impact

baseline = kpis.set_index('metric_key')['value']
impact = calculate_business_impact(
    production_volume=int(baseline['test_products_scored']),
    failure_rate=float(baseline['historical_failure_rate']),
    alert_rate=float(baseline['test_alert_rate']),
    precision=float(baseline['model_precision']),
    intervention_effectiveness=0.25,
    cost_per_failure=500.0,
    cost_per_alert_review=20.0,
)
pd.Series(impact.as_dict())

## 8. Run the executive dashboard

```powershell
.\.venv\Scripts\streamlit.exe run app\phase12_executive_dashboard.py
```

The dashboard contains Executive Overview, Station Heatmap, Bottlenecks, SHAP Drivers, and Business Impact tabs.

## 9. Production handoff

For a real manufacturing engagement, connect the dashboard to governed production tables, replace relative timestamps with plant calendar timestamps, define approved cost inputs with finance and operations, track realized interventions and outcomes, and distinguish model alerts from confirmed quality events.

In [ ]:
connection.close()